# EDA — RailMe v0

Exploratory data analysis for the Amtrak ridership dataset.

Run `python src/parse_and_join.py && python src/features.py` first to generate `data/processed/stations.csv`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from data_loader import load_stations, get_train_test_split

df = load_stations()
train, holdout = get_train_test_split(df)
print(f'Total: {len(df)}, Train: {len(train)}, Holdout: {len(holdout)}')
df.head()

## Ridership Distribution

In [ ]:
fig = px.histogram(
    df[df['annual_ridership'].notna()],
    x='annual_ridership',
    nbins=60,
    title='Annual Ridership Distribution (raw)',
    labels={'annual_ridership': 'Annual Ridership'},
    log_y=True,
)
fig.show()

In [ ]:
fig = px.histogram(
    df[df['log_ridership'].notna()],
    x='log_ridership',
    nbins=40,
    title='Log Ridership Distribution (log1p transformed)',
    labels={'log_ridership': 'log(1 + ridership)'},
)
fig.show()

## Top 20 Stations by Ridership

In [ ]:
name_col = 'station_name' if 'station_name' in df.columns else 'Station'
top20 = df[df['annual_ridership'].notna()].nlargest(20, 'annual_ridership')
fig = px.bar(
    top20, x='annual_ridership', y=name_col,
    orientation='h', title='Top 20 Amtrak Stations by Annual Ridership',
    labels={'annual_ridership': 'Annual Ridership'},
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## Feature Correlations with Log Ridership

In [ ]:
numeric_features = [
    'metro_pop', 'distance_to_nearest_major_city_km', 'num_nearby_stations',
    'modes_served', 'is_metro_area', 'is_northeast_corridor',
    'has_heavy_rail', 'has_commuter_rail', 'has_light_rail',
    'has_transit_bus', 'has_intercity_bus', 'has_air_connection', 'has_bikeshare',
]
numeric_features = [f for f in numeric_features if f in df.columns]

corr = df[numeric_features + ['log_ridership']].dropna(subset=['log_ridership']).corr()['log_ridership'].drop('log_ridership').sort_values()
fig = px.bar(
    x=corr.values, y=corr.index, orientation='h',
    title='Feature Correlation with log(Ridership)',
    labels={'x': 'Pearson r', 'y': 'Feature'},
)
fig.show()

## Geographic Map of Stations

In [ ]:
plot_df = df[df['lat'].notna() & df['lon'].notna() & df['annual_ridership'].notna()].copy()
fig = px.scatter_geo(
    plot_df,
    lat='lat', lon='lon',
    size='annual_ridership',
    color='log_ridership',
    hover_name=name_col,
    scope='usa',
    title='Amtrak Station Ridership (size = ridership)',
    color_continuous_scale='Viridis',
)
fig.show()

## Ridership vs. Metro Population

In [ ]:
scatter_df = df[df['metro_pop'].notna() & df['annual_ridership'].notna()].copy()
fig = px.scatter(
    scatter_df,
    x='metro_pop', y='annual_ridership',
    hover_name=name_col,
    color='is_northeast_corridor',
    log_x=True, log_y=True,
    title='Ridership vs Metro Population (log-log)',
    labels={'metro_pop': 'Metro Population', 'annual_ridership': 'Annual Ridership'},
)
fig.show()

## Missing Data Summary

In [ ]:
all_feats = numeric_features + ['station_type', 'annual_ridership']
missing = df[all_feats].isna().sum().sort_values(ascending=False)
print('Missing values per column:')
print((missing / len(df) * 100).round(1).to_string())